# Sentiment Analysis
- Using BERT


#### Student: Eric Michel
September 13, 2024


In [4]:
# !pip install transformers datasets torch

#!pip install accelerate -U

# !pip install transformers[torch]

# !pip install tf-keras

In [1]:
import pandas as pd
from datasets import Dataset

In [6]:
# Load dataset
df = pd.read_csv('./yelp.csv')

df.columns


Index(['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id',
       'cool', 'useful', 'funny', 'sentiment'],
      dtype='object')

In [7]:

df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,sentiment
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0,positive
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0,positive
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0,positive
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0,positive
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0,positive


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   business_id  10000 non-null  object
 1   date         10000 non-null  object
 2   review_id    10000 non-null  object
 3   stars        10000 non-null  int64 
 4   text         10000 non-null  object
 5   type         10000 non-null  object
 6   user_id      10000 non-null  object
 7   cool         10000 non-null  int64 
 8   useful       10000 non-null  int64 
 9   funny        10000 non-null  int64 
 10  sentiment    10000 non-null  object
dtypes: int64(4), object(7)
memory usage: 859.5+ KB


In [9]:
df.sentiment.unique()

array(['positive', 'negative', 'neutral'], dtype=object)

In [10]:
df.sentiment.value_counts()

,count
sentiment,
positive,8950
negative,921
neutral,129


In [11]:
# Check for any missing or non-string values in the 'text' column
print(df['text'].isnull().sum())  # Check for null values
print(df['text'].apply(lambda x: isinstance(x, str)).sum())  # Check for non-string entries


0
10000


In [12]:
# Drop rows where 'text' is null
df = df.dropna(subset=['text'])

# Ensure all values in the 'text' column are strings
df['text'] = df['text'].astype(str)


In [13]:
# Convert the dataset to a Hugging Face `Dataset`
# Ensure the `sentiment` column is mapped to integers (0 for negative, 1 for positive)
df['sentiment_label'] = df['sentiment'].map({'negative': 0, 'neutral': 1, 'positive': 2})

# Remove unnecessary columns
clean_dataset = df[['text','sentiment_label']]

dataset = Dataset.from_pandas(clean_dataset)

# Peek at the dataset to ensure it's loaded correctly
print(dataset)

Dataset({
    features: ['text', 'sentiment_label'],
    num_rows: 10000
})


# Tokenize data using BERT Tokenizer

In [14]:
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function to apply to each example
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

# Prepare data for PyTorch Training

In [15]:
# Rename the sentiment_label column to 'labels' since this is mandatory for PyTorch
tokenized_dataset = tokenized_dataset.rename_column('sentiment_label', 'labels')

# Set the dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')


# Split dataset to train and test

In [16]:
# Split the dataset into training and testing sets (80% train, 20% test)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 8000
})
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [17]:
train_dataset['labels']

tensor([2, 2, 2,  ..., 2, 2, 0])

# Load the model

In [18]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT for sequence classification (with 3 sentiment labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Define training parameters and initialize trainer

In [19]:
from transformers import Trainer, TrainingArguments


num_epocs = 10

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=num_epocs,      # number of epochs
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for logs
    logging_steps=10,
)

# Initialize the trainer
trainer = Trainer(
    model=model,                     # the pre-trained BERT model
    args=training_args,              # training arguments
    train_dataset=train_dataset,     # training dataset
    eval_dataset=test_dataset        # evaluation dataset
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Train Model / Fine-Tune BERT

In [20]:
import time

# Start time
start_time = time.time()
start_time

In [21]:
# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.136400,0.218766
2,0.206700,0.308878
3,0.176500,0.312304
4,0.007300,0.399364
5,0.033100,0.383197


TrainOutput(global_step=2500, training_loss=0.10454686366114765, metrics={'train_runtime': 4089.6806, 'train_samples_per_second': 9.781, 'train_steps_per_second': 0.611, 'total_flos': 1.052453670912e+16, 'train_loss': 0.10454686366114765, 'epoch': 5.0})

In [22]:

# End time
end_time = time.time()

# Calculate the duration in seconds
duration_seconds = end_time - start_time

# Convert the duration to minutes
duration_minutes = duration_seconds / 60

# Display the duration in minutes
print(f"End time: {end_time}")
print(f"Process duration: {duration_minutes:.2f} minutes")

Process duration: 68.18 minutes


# Evaluate Model

In [24]:
# Evaluate the model
results = trainer.evaluate()
print(results)


{'eval_loss': 0.38319671154022217, 'eval_runtime': 62.2447, 'eval_samples_per_second': 32.131, 'eval_steps_per_second': 2.008, 'epoch': 5.0}


# Save Model

In [26]:
# Save the model and tokenizer
model.save_pretrained('./sentiment_analysis_bert')
tokenizer.save_pretrained('./sentiment_analysis_bert')


('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/special_tokens_map.json',
 './fine_tuned_model/vocab.txt',
 './fine_tuned_model/added_tokens.json')

In [27]:
!zip mymodel.zip sentiment_analysis_bert/

  adding: fine_tuned_model/ (stored 0%)


In [28]:
import shutil

# Specify the folder you want to zip and the output zip file name
folder_to_zip = './sentiment_analysis_bert'  # Replace with the folder you want to zip
output_zip_file = './models/sentiment_analysis_bert.zip'         # zip file name

# Create the zip file
shutil.make_archive(output_zip_file.replace('.zip', ''), 'zip', folder_to_zip)

print(f"Folder '{folder_to_zip}' has been successfully zipped into '{output_zip_file}'")


Folder './fine_tuned_model' has been successfully zipped into './sample_data/fine_tuned_model.zip'


# Using Saved Model

In [2]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch


# Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained('./models/sentiment_analysis_bert')

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./models/sentiment_analysis_bert')


In [5]:

## Example text to classify sentiment

# text = "The movie was amazing and I loved it!"
# text = "The movie was fantastic!"  # Expected output: 'positive'
# text = "It was okay, not the best but not the worst either."  # Expected output: 'neutral'
text = "The movie was bad."  # Expected output: 'negative'



# Tokenize the input text (same as how it was done during training)
inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True)

In [6]:
# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")


Predicted sentiment: Negative
